<span style="font-size: 28px;">__Joint Diagnostic-Profile Narrowing, Joint Coverage, and Test-Set Denominators__</span>

----

**Purpose:** Address the JHIR minor-revision requests:

| Editor point | Request | Section below |
|---|---|---|
| 1 | Empirical frequency with which the three label-level prediction sets *jointly* narrow the four-profile diagnostic space, over all 100 runs, stratified by CP method and α (supplementary table) | **Table S9** |
| 2 | Empirical *joint* (simultaneous three-label) coverage, and how logically inconsistent label combinations are handled | **Table S10a / S10b**, logical-consistency definitions |
| 4 | Test-set size per run (manuscript states a 60/20/20 split; Supplementary Tables 7–8 imply ≈3,720 test participants per run) | **Test-set denominators** |

**Inputs** (per-sample result CSVs in `{WORKSPACE_BUCKET}/data/dfs/`, multi-label, with demographic covariates; the same configuration reported in the manuscript and used by Notebook 8):
- Naïve and Class-Conditional ICP (Notebook 3): one row per (run × example × label), with stored per-class p-values, so prediction sets can be rebuilt at any α.
- CRC / RCPS (Notebook 4, MAPIE `MapieMultiLabelClassifier`): one row per (run × example × label), with a boolean indicating whether the label is in the prediction set, at α = 0.1 and α = 0.25 (separate files).

**Outputs:** Run-level summary tables (mean ± SD across the 100 runs, with the 2.5th–97.5th percentile across runs) written to `{WORKSPACE_BUCKET}/data/dfs/nb9__*.csv`. No participant-level data are exported.

**All of Us Data and Statistics Dissemination Policy:** any value derived from a pooled count of 1–20 observations is suppressed in the displayed and exported tables (see `suppress_small_cells`). Check the exported tables before taking them out of the Researcher Workbench.

----

<span style="font-size: 22px;">__Definitions__</span>

**Labels** (`label_idx`): 0 = Anxiety, 1 = Depression, 2 = No Disorder. Each is an independent binary output of the LSTM.

**Valid diagnostic profiles.** These are the four logically consistent (Anxiety, Depression, No Disorder) vectors:

| Profile | Anxiety | Depression | No Disorder |
|---|---|---|---|
| No Disorder | 0 | 0 | 1 |
| Anxiety Only | 1 | 0 | 0 |
| Depression Only | 0 | 1 | 0 |
| Comorbid | 1 | 1 | 0 |

**Compatible profiles.** For each test observation, a profile is *compatible* with the model output when the output does not exclude it:

- **ICP (Naïve and Class-Conditional).** Each label *l* has its own prediction set *S<sub>l</sub>* ⊆ {Absent, Present}. A profile is compatible if, for every label, the profile's value for that label is in *S<sub>l</sub>*. The raw label-level sets are **not modified**: no repair, no re-coding, no dropping of observations.
- **CRC / RCPS.** The output is one set *C* ⊆ {Anxiety, Depression, No Disorder} of labels that the method declines to rule out. It is calibrated to control recall loss, 1 − |C ∩ Y| / |Y|. Labels outside *C* are treated as ruled out. A profile is compatible if all of its *present* labels are in *C*.

**Number of compatible profiles (0–4):**
- 4: no narrowing
- 2–3: partial narrowing
- 1: resolution to a single profile ("correct" if it is the participant's true profile)
- 0: **logically inconsistent / empty**. For ICP this happens when the label-level sets cannot be jointly satisfied by any valid profile, for example *S*<sub>No Disorder</sub> = {Present} with *S*<sub>Anxiety</sub> = {Present}, or when any label-level set is empty. For CRC/RCPS it happens only when *C* is empty. These observations are counted and reported, never discarded.

**Joint (simultaneous) coverage.**
- ICP: the true value of **all three** labels lies in the corresponding label-level sets.
- CRC/RCPS: every truly present label is in *C* (sample-level recall = 1).

When the true label vector is a valid profile, joint coverage is equivalent to "the true profile is among the compatible profiles". The code asserts this.

**Run-level aggregation.** Each indicator is averaged within each run and then summarised across the 100 runs as mean ± SD [2.5th–97.5th percentile].

<span style="font-size: 22px;">__Setup__</span>

In [ ]:
import os
import ast
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)
pd.set_option('display.max_colwidth', 60)

In [ ]:
import glob

# Workbench 1.0 sets WORKSPACE_BUCKET (gs://...). Workbench 2.0 does not; it mounts workspace
# buckets as folders under ~/workspace/<resource-name>/ (visible in the Jupyter file browser).
my_bucket = os.getenv('WORKSPACE_BUCKET')
if my_bucket:
    dfs_dir = f'{my_bucket}/data/dfs/'
else:
    candidates = sorted(set(glob.glob(os.path.expanduser('~/workspace/*/data/dfs/')) +
                            glob.glob('/home/*/workspace/*/data/dfs/')))
    assert len(candidates) == 1, f'expected one mounted data/dfs folder, found: {candidates}'
    dfs_dir = candidates[0]

# NB9_DATA_PREFIX / NB9_OUT_PREFIX override both (offline testing, or to hardcode a path)
prefix     = os.getenv('NB9_DATA_PREFIX', dfs_dir)
out_prefix = os.getenv('NB9_OUT_PREFIX',  dfs_dir)
print('reading from:', prefix)
print('writing to:  ', out_prefix)

ICP_FILES = {
    'Naïve ICP':             f'{prefix}results_df__multi-label-as-binary__not-class-conditional__inverse-probability_100x__per_run_timespans__no_leak__w_demographic_covariates.csv',
    'Class-Conditional ICP': f'{prefix}results_df__multi-label-as-binary__class-conditional__inverse-probability_100x__per_run_timespans__no_leak__w_demographic_covariates.csv',
}

# MAPIE CRC/RCPS results were produced in separate runs per alpha (Notebook 4)
MAPIE_FILES = {
    0.1:  f'{prefix}results_df__multi-label__mapie_100x__per_run_timespans__no_leak__w_demographic_covariates.csv',
    0.25: f'{prefix}results_df__multi-label__mapie_100x__per_run_timespans__no_leak__w_demographic_covariates__025_error_rate.csv',
}

ALPHAS = [0.1, 0.25]

LABELS = {0: 'Anxiety', 1: 'Depression', 2: 'No Disorder'}
L_IDX  = sorted(LABELS)

# (Anxiety, Depression, No Disorder)
PROFILES = {
    'No Disorder':     (0, 0, 1),
    'Anxiety Only':    (1, 0, 0),
    'Depression Only': (0, 1, 0),
    'Comorbid':        (1, 1, 0),
}
PROFILE_BY_VECTOR = {v: k for k, v in PROFILES.items()}

METHOD_ORDER = ['Naïve ICP', 'Class-Conditional ICP', 'CRC', 'RCPS']

# All of Us small-cell threshold (counts 1..20 may not be displayed)
SMALL_CELL_MAX = 20

## Load & Reshape ICP Results

In [ ]:
def load_icp(csv_path):
    """Load an ICP per-sample CSV (Notebook 3) and extract per-class p-values.

    prediction_p stores a list of (p_value, class_value) tuples; a class is in the
    prediction set at significance alpha iff p_value > alpha (PredictionClass.classes()).
    """
    df = pd.read_csv(csv_path, usecols=['run', 'example_idx', 'label_idx', 'person_id',
                                        'y_true', 'prediction_p', 'conformal_pred_and_alpha'])
    pmap = df['prediction_p'].map(ast.literal_eval).map(lambda pairs: {int(c): float(p) for p, c in pairs})
    df['p_absent']  = pmap.map(lambda d: d[0])
    df['p_present'] = pmap.map(lambda d: d[1])
    df['y_true'] = df['y_true'].astype(float).astype(int)
    df['label_idx'] = df['label_idx'].astype(int)
    df['run'] = df['run'].astype(int)
    df['example_idx'] = df['example_idx'].astype(int)
    print(f'Loaded {len(df):,} rows from {os.path.basename(csv_path)}; '
          f'runs = {df["run"].nunique()}, labels = {sorted(df["label_idx"].unique().tolist())}')
    return df


def to_wide(df, value_cols):
    """One row per (run, example_idx); columns '<col>_<label_idx>'. Verifies person_id agrees across labels."""
    key = ['run', 'example_idx', 'label_idx']
    dup = df.duplicated(key).sum()
    assert dup == 0, f'{dup} duplicated (run, example_idx, label_idx) rows'

    w = df.set_index(key)[value_cols + ['person_id']].unstack('label_idx')
    pid = w['person_id']
    assert pid.notna().all().all(), 'some (run, example) rows are missing a label'
    assert (pid.nunique(axis=1) == 1).all(), 'person_id differs across labels of the same example'

    out = pd.DataFrame(index=w.index)
    out['person_id'] = pid.iloc[:, 0]
    for c in value_cols:
        for l in L_IDX:
            out[f'{c}_{l}'] = w[(c, l)]
    return out.reset_index()

In [ ]:
icp_long = {m: load_icp(p) for m, p in ICP_FILES.items()}

In [ ]:
# Sanity check: rebuilding the stored alpha=0.1 prediction sets from p-values reproduces them exactly
for m, df in icp_long.items():
    stored = df['conformal_pred_and_alpha'].map(ast.literal_eval)
    stored_alpha = stored.map(lambda t: t[1]).unique()
    assert len(stored_alpha) == 1, stored_alpha
    a = float(stored_alpha[0])
    stored_set = stored.map(lambda t: frozenset(int(c) for c in t[0]))
    rebuilt = [frozenset(([0] if pa > a else []) + ([1] if pp > a else []))
               for pa, pp in zip(df['p_absent'], df['p_present'])]
    match = (stored_set.values == np.array(rebuilt, dtype=object)).mean() * 100
    print(f'{m}: rebuilt sets at alpha={a} match stored sets for {match:.2f}% of rows')
    assert match == 100.0

In [ ]:
icp_wide = {m: to_wide(df, ['y_true', 'p_absent', 'p_present']) for m, df in icp_long.items()}
for m, w in icp_wide.items():
    print(f'{m}: {len(w):,} (run × participant) observations')

## Load & Reshape CRC / RCPS Results

In [ ]:
def to_bool(s):
    if s.dtype == bool:
        return s
    return s.astype(str).str.strip().str.lower().map({'true': True, 'false': False, '1': True, '0': False,
                                                       '1.0': True, '0.0': False}).astype(bool)


def load_mapie(csv_path):
    df = pd.read_csv(csv_path, usecols=['run', 'example_idx', 'label_idx', 'person_id', 'y_true', 'crc', 'rcps'])
    for c in ['crc', 'rcps']:
        df[c] = to_bool(df[c])
    df['y_true'] = df['y_true'].astype(float).astype(int)
    for c in ['run', 'example_idx', 'label_idx']:
        df[c] = df[c].astype(int)
    print(f'Loaded {len(df):,} rows from {os.path.basename(csv_path)}; '
          f'runs = {df["run"].nunique()}, labels = {sorted(df["label_idx"].unique().tolist())}')
    return df

In [ ]:
mapie_wide = {a: to_wide(load_mapie(p), ['y_true', 'crc', 'rcps']) for a, p in MAPIE_FILES.items()}

## Sanity Checks: Label Order and Observed True Label Vectors

The profile logic assumes `label_idx` 0/1/2 = Anxiety/Depression/No Disorder in every file.
The checks below:
- tabulate the observed true (Anxiety, Depression, No Disorder) vectors;
- confirm that the ICP and MAPIE files agree on each participant's true labels.

In [ ]:
def true_vector_table(w, name):
    vec = list(zip(*[w[f'y_true_{l}'] for l in L_IDX]))
    s = pd.Series([PROFILE_BY_VECTOR.get(v, f'INVALID {v}') for v in vec]).value_counts()
    print(f'--- {name}: true (Anx, Dep, NoDis) vectors (pooled over runs) ---')
    print(s.to_string()); print()

true_vector_table(icp_wide['Naïve ICP'], 'Naïve ICP file')
true_vector_table(mapie_wide[0.1], 'MAPIE alpha=0.1 file')

# same participant -> same true labels across files
y_cols = [f'y_true_{l}' for l in L_IDX]
a = icp_wide['Naïve ICP'].drop_duplicates('person_id').set_index('person_id')[y_cols]
for alpha, w in mapie_wide.items():
    b = w.drop_duplicates('person_id').set_index('person_id')[y_cols]
    common = a.index.intersection(b.index)
    agree = (a.loc[common].values == b.loc[common].values).all(axis=1).mean() * 100
    print(f'ICP vs MAPIE (alpha={alpha}): {len(common):,} shared participants; true label vectors agree for {agree:.2f}%')

## Editor Point 4 — Test-Set Denominators

`get_scaled_reshaped_train_calibrate_test` (Notebooks 3–5) splits **unique participants**:

```python
person_ids_train, person_ids_temp = train_test_split(person_ids, test_size=0.6, ...)       # 40% train
person_ids_calibration, person_ids_test = train_test_split(person_ids_temp, test_size=0.5, ...)  # 30% cal / 30% test
```

The implemented split is therefore **40% / 30% / 30%** (train / calibration / test), not the 60/20/20 stated in the manuscript.
The cells below report the realised test-set size per run. They also check that it matches a 30% test fraction of the run-specific eligible sample (12,371–12,443 across runs; Supplementary Material).

In [ ]:
from math import ceil

def split_sizes(n):
    """Exact sklearn train_test_split sizes for the implemented two-stage split (test_size = ceil(frac * n))."""
    n_temp = ceil(0.6 * n)
    n_test = ceil(0.5 * n_temp)
    return n - n_temp, n_temp - n_test, n_test

def implied_eligible_n(n_test, lo=12000, hi=13000):
    return [n for n in range(lo, hi) if split_sizes(n)[2] == n_test]

rows = []
sources = {**{m: w for m, w in icp_wide.items()},
           **{f'CRC/RCPS (α={a})': w for a, w in mapie_wide.items()}}
for name, w in sources.items():
    n = w.groupby('run').size()
    n_unique = w.groupby('run')['person_id'].nunique()
    assert (n == n_unique).all(), f'{name}: duplicated participants within a run'
    implied = n.map(lambda k: implied_eligible_n(k))
    rows.append({
        'Source': name,
        'Runs': n.size,
        'Test participants / run (mean ± SD)': f'{n.mean():,.1f} ± {n.std():.1f}',
        'Min–Max': f'{n.min():,}–{n.max():,}',
        'Implied eligible N (min–max)': f'{min(min(v) for v in implied):,}–{max(max(v) for v in implied):,}',
        'Test fraction (vs. implied N)': f'{(n / implied.map(np.mean)).mean():.3f}',
        'Participant-label rows / run': f'{3 * n.mean():,.0f}',
    })
denom_tbl = pd.DataFrame(rows)
display(denom_tbl)

for n in (12371, 12405, 12420, 12443):
    tr, ca, te = split_sizes(n)
    print(f'N eligible = {n:,}: implemented 40/30/30 -> train {tr:,}, calibration {ca:,}, test {te:,};   '
          f'a 60/20/20 split would give test ≈ {round(0.2 * n):,}')

In [ ]:
# Same test participants in every file for a given run (same seed => same split)?
ref = icp_wide['Naïve ICP'].groupby('run')['person_id'].apply(frozenset)
for name, w in sources.items():
    other = w.groupby('run')['person_id'].apply(frozenset)
    common_runs = ref.index.intersection(other.index)
    same = np.mean([ref[r] == other[r] for r in common_runs]) * 100
    print(f'{name:28s}: identical test-participant set to Naïve ICP in {same:.0f}% of {len(common_runs)} runs')

In [ ]:
# Reconcile with Supplementary Tables 7-8: per label and run, Absent + Present = test N,
# and singleton + full + empty set counts = test N
w = icp_wide['Naïve ICP']
for l in L_IDX:
    n_run = w.groupby('run').size()
    pres = w.groupby('run')[f'y_true_{l}'].sum()
    print(f'{LABELS[l]:12s}: mean Present / run = {pres.mean():7.1f}, mean Absent / run = {(n_run - pres).mean():7.1f}, '
          f'sum = {n_run.mean():7.1f}')

## Observation-Level Joint Evaluation

In [ ]:
def _profile_compat_icp(in_abs, in_pre):
    compat = {}
    for name, vec in PROFILES.items():
        ok = np.ones(in_abs.shape[0], dtype=bool)
        for j, v in enumerate(vec):
            ok &= in_pre[:, j] if v == 1 else in_abs[:, j]
        compat[name] = ok
    return compat


def _profile_compat_set(flag):
    """flag[:, j] = label j is in the CRC/RCPS set C. A profile is compatible if its present labels are all in C."""
    compat = {}
    for name, vec in PROFILES.items():
        ok = np.ones(flag.shape[0], dtype=bool)
        for j, v in enumerate(vec):
            if v == 1:
                ok &= flag[:, j]
        compat[name] = ok
    return compat


def _common(w, method, alpha, compat, joint_cov):
    Y = w[[f'y_true_{l}' for l in L_IDX]].to_numpy().astype(int)
    true_profile = np.array([PROFILE_BY_VECTOR.get(tuple(v), 'Invalid') for v in Y], dtype=object)
    C = np.column_stack([compat[p] for p in PROFILES])
    n_compat = C.sum(axis=1)

    true_compat = np.zeros(len(w), dtype=bool)
    for i, p in enumerate(PROFILES):
        true_compat |= (true_profile == p) & C[:, i]

    valid = true_profile != 'Invalid'
    assert (joint_cov[valid] == true_compat[valid]).all(), 'joint coverage != true-profile compatibility'

    obs = pd.DataFrame({
        'method': method, 'alpha': alpha,
        'run': w['run'].to_numpy(), 'person_id': w['person_id'].to_numpy(),
        'true_profile': true_profile,
        'any_positive_dx': (Y[:, 0] == 1) | (Y[:, 1] == 1),
        'n_compat': n_compat,
        'joint_cov': joint_cov,
        'true_profile_retained': true_compat,
        'retained_profiles': [' + '.join(p for p, k in zip(PROFILES, row) if k) or '(none)' for row in C],
    })
    for k in range(5):
        obs[f'compat_{k}'] = n_compat == k
    obs['any_narrowing']      = (n_compat >= 1) & (n_compat <= 3)
    obs['narrowing_ge_50pct'] = (n_compat >= 1) & (n_compat <= 2)
    obs['resolved_single']    = n_compat == 1
    obs['resolved_correct']   = (n_compat == 1) & true_compat
    obs['inconsistent_or_empty'] = n_compat == 0
    return obs, Y


def evaluate_icp(w, method, alpha):
    p_abs = w[[f'p_absent_{l}'  for l in L_IDX]].to_numpy()
    p_pre = w[[f'p_present_{l}' for l in L_IDX]].to_numpy()
    in_abs, in_pre = p_abs > alpha, p_pre > alpha
    Y = w[[f'y_true_{l}' for l in L_IDX]].to_numpy().astype(int)
    label_cov = np.where(Y == 1, in_pre, in_abs)
    joint_cov = label_cov.all(axis=1)

    obs, _ = _common(w, method, alpha, _profile_compat_icp(in_abs, in_pre), joint_cov)
    set_size = in_abs.astype(int) + in_pre.astype(int)
    for j, l in enumerate(L_IDX):
        obs[f'cov_{LABELS[l]}'] = label_cov[:, j]
    obs['any_label_empty'] = (set_size == 0).any(axis=1)
    return obs


def evaluate_set(w, method_col, method, alpha):
    flag = w[[f'{method_col}_{l}' for l in L_IDX]].to_numpy().astype(bool)
    Y = w[[f'y_true_{l}' for l in L_IDX]].to_numpy().astype(int)
    joint_cov = (flag | (Y == 0)).all(axis=1)

    obs, _ = _common(w, method, alpha, _profile_compat_set(flag), joint_cov)
    for j, l in enumerate(L_IDX):
        # label-level recall among participants for whom the label is truly present (NaN otherwise)
        obs[f'recall_{LABELS[l]}'] = np.where(Y[:, j] == 1, flag[:, j], np.nan)
    obs['sample_recall'] = (flag & (Y == 1)).sum(axis=1) / Y.sum(axis=1)   # MAPIE requires >=1 positive label
    obs['set_size'] = flag.sum(axis=1)
    return obs

In [ ]:
obs_parts = []
for alpha in ALPHAS:
    for m, w in icp_wide.items():
        obs_parts.append(evaluate_icp(w, m, alpha))
    obs_parts.append(evaluate_set(mapie_wide[alpha], 'crc',  'CRC',  alpha))
    obs_parts.append(evaluate_set(mapie_wide[alpha], 'rcps', 'RCPS', alpha))
obs = pd.concat(obs_parts, ignore_index=True)
obs['method'] = pd.Categorical(obs['method'], METHOD_ORDER, ordered=True)
print(f'{len(obs):,} (method × alpha × run × participant) observations')
obs.groupby(['method', 'alpha'], observed=True).size()

## Run-Level Summary Helpers

In [ ]:
def summarize(df, by, indicators, continuous=()):
    """Per-run mean of each indicator (NaN-aware), summarised across runs.

    Returns a tidy frame: one row per (group, indicator) with mean/sd/p2.5/p97.5 of the per-run
    percentages, plus pooled numerator/denominator (used only for small-cell suppression).
    """
    rows = []
    for keys, g in df.groupby(by, observed=True, sort=True):
        keys = keys if isinstance(keys, tuple) else (keys,)
        n_run = g.groupby('run').size()
        for ind in indicators:
            x = g[ind].astype(float)
            per_run = x.groupby(g['run']).mean() * 100
            num = np.nan if ind in continuous else float(np.nansum(x))
            den = float(x.notna().sum())
            rows.append({**dict(zip(by, keys)),
                         'indicator': ind,
                         'n_runs': per_run.notna().sum(),
                         'mean_n_per_run': (x.notna().groupby(g['run']).sum()).mean(),
                         'mean_pct': per_run.mean(), 'sd_pct': per_run.std(),
                         'p2.5_pct': per_run.quantile(0.025), 'p97.5_pct': per_run.quantile(0.975),
                         'pooled_numerator': num, 'pooled_denominator': den})
    return pd.DataFrame(rows)


def suppress_small_cells(tidy):
    """Flag cells whose pooled numerator, or its complement, is 1..SMALL_CELL_MAX observations."""
    t = tidy.copy()
    num, den = t['pooled_numerator'], t['pooled_denominator']
    comp = den - num
    small = (num.between(1, SMALL_CELL_MAX)) | (comp.between(1, SMALL_CELL_MAX)) | den.between(1, SMALL_CELL_MAX)
    t['suppressed'] = small.fillna(False)
    for c in ['mean_pct', 'sd_pct', 'p2.5_pct', 'p97.5_pct', 'pooled_numerator']:
        t.loc[t['suppressed'], c] = np.nan
    return t


def fmt_cell(r, dec=1):
    if r.get('suppressed', False):
        return f'<{SMALL_CELL_MAX + 1} obs (suppressed)'
    if pd.isna(r['mean_pct']):
        return '—'
    return f"{r['mean_pct']:.{dec}f} ± {r['sd_pct']:.{dec}f} [{r['p2.5_pct']:.{dec}f}–{r['p97.5_pct']:.{dec}f}]"


def to_display(tidy, by, columns):
    t = tidy.copy()
    t['cell'] = t.apply(fmt_cell, axis=1)
    wide = t.pivot_table(index=by, columns='indicator', values='cell', aggfunc='first', observed=True)
    wide = wide[[c for c in columns if c in wide.columns]].rename(columns=columns)
    wide.columns.name = None
    wide = wide.reset_index()
    wide['method'] = pd.Categorical(wide['method'].astype(str), METHOD_ORDER, ordered=True)
    sort_cols = [c for c in ['method', 'alpha', 'true_profile'] if c in wide.columns]
    if 'true_profile' in wide.columns:
        wide['true_profile'] = pd.Categorical(wide['true_profile'].astype(str), list(PROFILES), ordered=True)
    return wide.sort_values(sort_cols).reset_index(drop=True)

## Table S9 — Joint Diagnostic-Profile Narrowing (Editor Point 1)

Percentage of test observations by number of compatible profiles. Values are mean ± SD [2.5th–97.5th percentile] across 100 runs.
- *Any narrowing* = 1–3 compatible profiles.
- *≥50% narrowing* = 1–2 compatible profiles.
- *Resolved & correct* = exactly one compatible profile, and it is the true profile.

In [ ]:
S9_COLS = {
    'compat_0': 'Zero (inconsistent / empty), %',
    'compat_1': 'One profile, %',
    'compat_2': 'Two profiles, %',
    'compat_3': 'Three profiles, %',
    'compat_4': 'Four profiles (no narrowing), %',
    'any_narrowing': 'Any narrowing, %',
    'narrowing_ge_50pct': '≥50% narrowing, %',
    'resolved_correct': 'Resolved to single profile & correct, %',
}
s9_tidy = suppress_small_cells(summarize(obs, ['method', 'alpha'], list(S9_COLS)))
s9 = to_display(s9_tidy, ['method', 'alpha'], S9_COLS)
display(s9)

### Table S9 (supplement) — Narrowing Stratified by True Diagnostic Profile

This addresses the informativeness for participants with positive diagnoses raised in editor point 5 (c).

In [ ]:
S9B_COLS = {
    'compat_4': 'Four profiles (no narrowing), %',
    'any_narrowing': 'Any narrowing, %',
    'true_profile_retained': 'True profile retained (joint coverage), %',
    'resolved_single': 'Resolved to single profile, %',
    'resolved_correct': 'Resolved to single profile & correct, %',
    'compat_0': 'Zero (inconsistent / empty), %',
}
obs_valid = obs[obs['true_profile'] != 'Invalid'].copy()
obs_valid['true_profile'] = pd.Categorical(obs_valid['true_profile'], list(PROFILES), ordered=True)
s9b_tidy = suppress_small_cells(summarize(obs_valid, ['method', 'alpha', 'true_profile'], list(S9B_COLS)))
s9b = to_display(s9b_tidy, ['method', 'alpha', 'true_profile'], S9B_COLS)
n_per_profile = (obs_valid.groupby(['method', 'alpha', 'true_profile', 'run'], observed=True).size()
                 .groupby(level=['method', 'alpha', 'true_profile'], observed=True).mean()
                 .rename('Mean N / run').round(1).reset_index())
s9b = n_per_profile.merge(s9b, on=['method', 'alpha', 'true_profile'])
display(s9b)

### Which profiles are retained? (most frequent compatible-profile sets)

In [ ]:
pattern = (obs.groupby(['method', 'alpha', 'retained_profiles'], observed=True).size()
              .rename('pooled_n').reset_index())
pattern['pct'] = pattern['pooled_n'] / pattern.groupby(['method', 'alpha'], observed=True)['pooled_n'].transform('sum') * 100
pattern = pattern[pattern['pooled_n'] > SMALL_CELL_MAX]          # small cells not shown
pattern = (pattern.sort_values(['method', 'alpha', 'pct'], ascending=[True, True, False])
                  .groupby(['method', 'alpha'], observed=True).head(6))
pattern['pct'] = pattern['pct'].round(1)
display(pattern.drop(columns='pooled_n'))

## Table S10 — Label-Level vs. Joint Coverage and Logical Consistency (Editor Point 2)

**S10a — ICP.**
- Label-level marginal coverage: true value ∈ *S<sub>l</sub>*.
- Joint three-label coverage: all three true values are covered simultaneously.
- Logically inconsistent: no valid profile is compatible. Of these, *any label set empty* is a subset.

ICP gives a nominal 1 − α guarantee for each label separately (marginal for Naïve ICP; per-class for Class-Conditional ICP). It does **not** give a nominal 1 − α guarantee for the complete three-label vector.

In [ ]:
S10A_COLS = {
    'cov_Anxiety': 'Anxiety coverage, %',
    'cov_Depression': 'Depression coverage, %',
    'cov_No Disorder': 'No Disorder coverage, %',
    'joint_cov': 'Joint three-label coverage, %',
    'inconsistent_or_empty': 'Logically inconsistent (0 profiles), %',
    'any_label_empty': '…of which ≥1 empty label set, %',
}
icp_obs = obs[obs['method'].isin(['Naïve ICP', 'Class-Conditional ICP'])]
s10a_tidy = suppress_small_cells(summarize(icp_obs, ['method', 'alpha'], list(S10A_COLS)))
s10a = to_display(s10a_tidy, ['method', 'alpha'], S10A_COLS)
display(s10a)

**S10b — CRC / RCPS.** These methods calibrate *sample-averaged recall* (1 − α on average for CRC; with probability ≥ 1 − δ, δ = 0.01, for RCPS, using MAPIE's default Waudby-Smith–Ramdas bound).
- Mean sample recall is the calibrated quantity.
- Label-level recall is computed among participants for whom that label is truly present.
- Joint coverage means every truly present label is in the set (sample recall = 1).
- An empty set is the only way to retain zero profiles.

In [ ]:
S10B_COLS = {
    'sample_recall': 'Mean sample recall (calibrated target), %',
    'recall_Anxiety': 'Anxiety recall (if present), %',
    'recall_Depression': 'Depression recall (if present), %',
    'recall_No Disorder': 'No Disorder recall (if present), %',
    'joint_cov': 'Joint coverage (all present labels in set), %',
    'inconsistent_or_empty': 'Empty set (0 profiles), %',
}
set_obs = obs[obs['method'].isin(['CRC', 'RCPS'])]
s10b_tidy = suppress_small_cells(summarize(set_obs, ['method', 'alpha'], list(S10B_COLS),
                                           continuous=('sample_recall',)))
s10b = to_display(s10b_tidy, ['method', 'alpha'], S10B_COLS)

# fraction of runs in which the aggregate recall target (>= 1 - alpha) was met on the test set
met = (set_obs.groupby(['method', 'alpha', 'run'], observed=True)['sample_recall'].mean().reset_index())
met['met'] = met['sample_recall'] >= 1 - met['alpha']
met = (met.groupby(['method', 'alpha'], observed=True)['met'].mean() * 100).rename('Runs meeting 1−α recall, %').reset_index()
s10b = s10b.merge(met, on=['method', 'alpha'])
display(s10b)

### Joint Coverage Stratified by True Profile (all methods)

In [ ]:
jc_tidy = suppress_small_cells(summarize(obs_valid, ['method', 'alpha', 'true_profile'], ['joint_cov']))
jc = to_display(jc_tidy, ['method', 'alpha'] + ['true_profile'], {'joint_cov': 'Joint coverage, %'})
display(jc.pivot(index=['method', 'alpha'], columns='true_profile', values='Joint coverage, %'))

## Headline Numbers for the Manuscript / Response Letter

In [ ]:
def headline(tidy, ind):
    t = tidy[tidy['indicator'] == ind]
    return {(r['method'], r['alpha']): (f"{r['mean_pct']:.1f}% ± {r['sd_pct']:.1f}" if not r['suppressed'] else 'suppressed')
            for _, r in t.iterrows()}

for label, tidy, ind in [('Any narrowing of the four-profile space', s9_tidy, 'any_narrowing'),
                         ('Resolved to a single correct profile', s9_tidy, 'resolved_correct'),
                         ('Logically inconsistent / empty (0 profiles)', s9_tidy, 'compat_0'),
                         ('Joint three-label coverage (ICP)', s10a_tidy, 'joint_cov'),
                         ('Joint coverage (CRC/RCPS)', s10b_tidy, 'joint_cov')]:
    print(f'--- {label} ---')
    for (m, a), v in headline(tidy, ind).items():
        print(f'  {m:24s} α={a:<5} {v}')
    print()

## Save Summary Tables
Only aggregated, small-cell-suppressed run-level summaries are written.

In [ ]:
outputs = {
    'nb9__test_set_denominators.csv':                    denom_tbl,
    'nb9__tableS9_joint_profile_narrowing.csv':          s9,
    'nb9__tableS9_by_true_profile.csv':                  s9b,
    'nb9__tableS10a_icp_joint_coverage.csv':             s10a,
    'nb9__tableS10b_crc_rcps_joint_coverage.csv':        s10b,
    'nb9__retained_profile_patterns.csv':                pattern.drop(columns='pooled_n'),
    'nb9__tidy_run_level_summaries.csv':                 pd.concat([s9_tidy, s9b_tidy, s10a_tidy, s10b_tidy], ignore_index=True)
                                                           .drop(columns=['pooled_numerator', 'pooled_denominator']),
}
for fname, df_out in outputs.items():
    df_out.to_csv(f'{out_prefix}{fname}', index=False)
    print(f'wrote {out_prefix}{fname}')